In [1]:
import os
lat = 34.499984
lon = -4.708586
site_name = 'test_site_name'
client = '34,499984_-4,708586'
surface_tilt = 30
startDate = "2005-01-01"
endDate   = "2025-12-31"
job_id = None  
selected_files = []


In [2]:
# Parameters
lat = 32.512851
lon = -3.881781
site_name = "Talsint-Morocco"
startDate = "2005-01-01"
endDate = "2024-12-31"
job_id = 29


In [3]:
# Status updater — uses HTTP so no Django setup needed
import requests

def update_job(status, message=''):
    if job_id:
        try:
            requests.post(
                #  CORRECT - Docker service name
                # f'http://web:8000/api/tmy/update/{job_id}/',
                f'http://web:8000/api/tmy/update-internal/{job_id}/',
                json={'status': status, 'message': message},
                timeout=5
            )
            print(f"[Job #{job_id}] {status}: {message}")
        except Exception as e:
            print(f"[update_job failed] {e}")
    else:
        print(f"[No job_id] {status}: {message}")

In [4]:
import pandas as pd
import numpy as np

In [5]:
TMYs_folder = 'TMYs'
TMYs_path = os.path.join(TMYs_folder,site_name)
if not os.path.exists(TMYs_path):
    os.makedirs(TMYs_path)

In [6]:
import os

downloads_dir = os.path.join(TMYs_folder, site_name, "downloads")
os.makedirs(downloads_dir, exist_ok=True)

print("Selected files:", selected_files)

Selected files: []


### Configuring cdsapi

In [7]:
api_token = {
    'era5': {
        'url': 'https://cds.climate.copernicus.eu/api'
    },
    'cams': {
        'url': 'https://ads.atmosphere.copernicus.eu/api'
    }
}
# def update_cdsapirc(api_type, file_path=r"C:\Users\DELL\.cdsapirc"):

# def update_cdsapirc(api_type, file_path=None):
#     if file_path is None:
#         file_path = os.path.expanduser("~/.cdsapirc")
#     """
#     Updates the .cdsapirc file with the specified API URL.

#     Parameters:
#         api_type (str): The API type to use ('era5' or 'cams').
#         file_path (str): The path to the .cdsapirc file.
#     """
#     # Define the API URLs
#     api_url = {
#         'era5': {
#             'url': 'https://cds.climate.copernicus.eu/api'
#         },
#         'cams': {
#             'url': 'https://ads.atmosphere.copernicus.eu/api'
#         }
#     }

#     # Validate the input
#     if api_type not in api_url:
#         raise ValueError("Invalid API type. Please choose 'era5' or 'cams'.")

#     try:
#         # Read the file content
#         with open(file_path, 'r') as file:
#             lines = file.readlines()

#         # Update the lines based on the selected API type
#         updated_lines = [] 
#         for line in lines:
#             if "url:" in line:
#                 updated_lines.append(f"url: {api_url[api_type]['url']}\n")
#             else:
#                 updated_lines.append(line)

#         # Write the updated content back to the file
#         with open(file_path, 'w') as file:
#             file.writelines(updated_lines)

#         print(f"The .cdsapirc file has been updated to use '{api_type}' API.")
#     except FileNotFoundError:
#         print(f"The file '{file_path}' does not exist.")
#     except Exception as e:
#         print(f"An error occurred: {e}")


def update_cdsapirc(api_type, file_path=None):
    if file_path is None:
        file_path = os.path.expanduser("~/.cdsapirc")
    
    api_url = {
        'era5': {'url': 'https://cds.climate.copernicus.eu/api'},
        'cams': {'url': 'https://ads.atmosphere.copernicus.eu/api'}
    }

    if api_type not in api_url:
        raise ValueError("Invalid API type. Choose 'era5' or 'cams'.")

    try:
        with open(file_path, 'r') as file:
            lines = file.readlines()

        updated_lines = []
        for line in lines:
            if "url:" in line:
                updated_lines.append(f"url: {api_url[api_type]['url']}\n")
            else:
                updated_lines.append(line)

        with open(file_path, 'w') as file:
            file.writelines(updated_lines)

        print(f"Updated .cdsapirc to use '{api_type}' API.")
    except FileNotFoundError:
        print(f"File not found: {file_path}")
    except Exception as e:
        print(f"Error: {e}")

In [8]:
update_cdsapirc('cams')

Updated .cdsapirc to use 'cams' API.


### Downloading cams data

In [9]:
import cdsapi

update_job('downloading_cams', 'Downloading CAMS solar radiation data from Copernicus ADS...')
c = cdsapi.Client()

c.retrieve(
    "cams-solar-radiation-timeseries",
    {
    "sky_type": "observed_cloud",
    "location": {"longitude": lon, "latitude": lat},
    "altitude": ["-999."],
    "date": [f'{startDate}/{endDate}'],
    "time_step": "1hour",
    "time_reference": "universal_time",
    "format": "csv"
    },
    f'{TMYs_folder}/{site_name}/{site_name}_cams.csv')
# # STATUS UPDATE
# if 'job_id' in vars() and job_id:
#     from tmy_app.models import TMYJob
#     j = TMYJob.objects.get(id=job_id)
#     j.status = 'processing_data'
#     j.status_message = 'All data downloaded. Processing and merging ERA5 + CAMS datasets...'
#     j.save()


[Job #29] downloading_cams: Downloading CAMS solar radiation data from Copernicus ADS...


2026-06-23 21:32:47,854 INFO Request ID is edd59977-aaa1-4acf-99e9-cb91ade99fb6


2026-06-23 21:32:47,955 INFO status has been updated to accepted


2026-06-23 21:33:11,529 INFO status has been updated to running


2026-06-23 21:35:42,704 INFO status has been updated to successful


...

2ef781b2678808617d2b7f1db36092d1.csv:  88%|████████▊ | 18.0M/20.5M [00:08<00:01, 1.48MB/s]

2ef781b2678808617d2b7f1db36092d1.csv:  93%|█████████▎| 19.0M/20.5M [00:09<00:01, 1.51MB/s]

2ef781b2678808617d2b7f1db36092d1.csv:  98%|█████████▊| 20.0M/20.5M [00:10<00:00, 1.57MB/s]

2ef781b2678808617d2b7f1db36092d1.csv: 100%|██████████| 20.5M/20.5M [00:10<00:00, 1.59MB/s]

'TMYs/Talsint-Morocco/Talsint-Morocco_cams.csv'

In [10]:
update_job('downloading_era5', f'Downloading ERA5 climate data {startDate[:4]} to {endDate[:4]}...')
update_cdsapirc('era5')
# # STATUS UPDATE
# if 'job_id' in vars() and job_id:
#     from tmy_app.models import TMYJob
#     j = TMYJob.objects.get(id=job_id)
#     j.status = 'downloading_era5'
#     j.status_message = 'Connecting to Copernicus CDS and downloading ERA5 data...'
#     j.save()

[Job #29] downloading_era5: Downloading ERA5 climate data 2005 to 2024...
Updated .cdsapirc to use 'era5' API.


### Extracting ERA5 Time series

In [11]:
import xarray as xr
import glob
from tqdm import tqdm
from metpy.calc import wind_speed, wind_direction, relative_humidity_from_dewpoint
from metpy.units import units

def round_to_quarter(value):
    return round(value / 0.25) * 0.25

lat_rounded = round_to_quarter(lat)
lon_rounded = round_to_quarter(lon)

c = cdsapi.Client()

c.retrieve(
    "reanalysis-era5-single-levels-timeseries",
    {
    "location": {"longitude": lon_rounded, "latitude": lat_rounded},
    "date": [f'{startDate}/{endDate}'],
    "data_format": "csv",
    "variable": [
        "2m_dewpoint_temperature",
        "mean_sea_level_pressure",
        "2m_temperature",
        "total_precipitation",
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "100m_u_component_of_wind",
        "100m_v_component_of_wind",
        "surface_solar_radiation_downwards"
    ],
    },
    f'{TMYs_folder}/{site_name}/{site_name}_era5.csv')
#     # STATUS UPDATE
# if 'job_id' in vars() and job_id:
#     from tmy_app.models import TMYJob
#     j = TMYJob.objects.get(id=job_id)
#     j.status = 'downloading_cams'
#     j.status_message = 'ERA5 download complete. Now downloading CAMS solar data...'
#     j.save()

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-06-23 21:35:55,396 INFO [2026-02-16T00:00:00] - To generate this ERA5 hourly time series dataset, **homogenisation conventions have been applied to the ERA5 source GRIB data** to ensure consistency, usability, and alignment across chosen variables and time steps. The processed data were then written to an **ARCO Zarr archive**, enabling efficient cloud-optimised access and scalable data retrieval. Please refer to the [user guide](https://confluence.ecmwf.int/x/R6cfHg) for details.

- The dataset presented here is a subset of selected parameters from the full [CDS ERA5 hourly data on single levels (1940–present)](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=overview). **Requirements for additional parameters may be considered**. Please raise your request with ECMWF Support [here](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202).


2026-06-23 21:35:55,398 INFO Request ID is 4706697f-0687-404f-a5aa-cc2babe720f4


2026-06-23 21:35:55,498 INFO status has been updated to accepted


2026-06-23 21:36:45,993 INFO status has been updated to running


...

339dec84d8201558bcb6270367668db7.zip:  15%|█▍        | 1.00M/6.68M [00:00<00:04, 1.44MB/s]

339dec84d8201558bcb6270367668db7.zip:  45%|████▍     | 3.00M/6.68M [00:00<00:00, 4.27MB/s]

339dec84d8201558bcb6270367668db7.zip:  60%|█████▉    | 4.00M/6.68M [00:01<00:00, 4.43MB/s]

339dec84d8201558bcb6270367668db7.zip:  75%|███████▍  | 5.00M/6.68M [00:01<00:00, 5.38MB/s]

'TMYs/Talsint-Morocco/Talsint-Morocco_era5.csv'

In [12]:
import zipfile
output_path = f"{TMYs_folder}/{site_name}/{site_name}_era5.csv"
import os

desired_name = f"{site_name}_era5.csv"
folder_path = os.path.dirname(output_path)

if zipfile.is_zipfile(output_path):
    print("ZIP detected. Extracting...")

    with zipfile.ZipFile(output_path, 'r') as z:
        extracted_files = z.namelist()
        z.extractall(folder_path)

    os.remove(output_path)

    # Rename extracted file to your desired name
    extracted_file_path = os.path.join(folder_path, extracted_files[0])
    final_path = os.path.join(folder_path, desired_name)

    os.rename(extracted_file_path, final_path)

    print(f"File renamed to: {desired_name}")

else:
    print("Normal CSV file.")


ZIP detected. Extracting...


File renamed to: Talsint-Morocco_era5.csv


In [13]:
era5_point = pd.read_csv(rf"{TMYs_folder}/{site_name}/{site_name}_era5.csv")

In [14]:
import glob
from tqdm import tqdm
from metpy.calc import wind_speed, wind_direction, relative_humidity_from_dewpoint
from metpy.units import units

era5_point['t2m_C'] = era5_point['t2m'] - 273.15
era5_point['d2m_C'] = era5_point['d2m'] - 273.15
era5_point['msl_hpa'] = era5_point['msl'] / 100

era5_point['wind_speed'] = wind_speed(era5_point['u10'].values * units.meter / units.second, 
                                    era5_point['v10'].values * units.meter / units.second).magnitude

era5_point['wind_direction'] = wind_direction(era5_point['u10'].values * units.meter / units.second, 
                                            era5_point['v10'].values * units.meter / units.second).magnitude

era5_point['relative_humidity'] = relative_humidity_from_dewpoint(
        era5_point['t2m_C'].values * units.degC,
        era5_point['d2m_C'].values * units.degC
    ).magnitude * 100  # Convert to percentage

filtred_era5_point = era5_point[['valid_time','t2m_C', 'd2m_C', 'wind_speed', 'wind_direction', 'relative_humidity', 'msl_hpa', 'ssrd']]

ERA5 = filtred_era5_point.rename(columns={
        'valid_time': 'time',
        't2m_C': 'Temperature',
        'd2m_C': 'Dew Point',
        'msl_hpa': 'Pressure',
        'relative_humidity': 'Relative Humidity',
        'wind_speed': 'Wind Speed',
        'wind_direction': 'Wind Direction',
        'ssrd': 'Surface solar radiation downwards'
    })

    # Save to CSV
ERA5.to_csv(f"{TMYs_folder}/{site_name}/era5_{site_name}.csv", index=False)
print("\nTime series extraction complete. Data saved.")


Time series extraction complete. Data saved.


In [15]:
era5 = pd.read_csv(rf"{TMYs_folder}/{site_name}/era5_{site_name}.csv")

In [16]:
if "ERA5" in selected_files:
    era5_point.to_csv(
        f"{downloads_dir}/ERA5.csv",
        index=False
    )

In [17]:
# Assuming your DataFrame is named df
era5['datetime'] = pd.to_datetime(era5['time'])

# Drop the original columns if you no longer need them
era5 = era5.drop(columns=['time'])

era5['datetime'] = pd.to_datetime(era5['datetime'])
era5.set_index('datetime', inplace = True)

In [18]:
era5

,Temperature,Dew Point,Wind Speed,Wind Direction,Relative Humidity,Pressure,Surface solar radiation downwards
datetime,,,,,,,
2005-01-01 00:00:00,0.47143,-5.97090,1.566414,318.199185,61.954797,1029.74940,0.0
2005-01-01 01:00:00,0.51060,-5.69320,1.570847,315.224321,63.101567,1029.34000,0.0
2005-01-01 02:00:00,-0.43156,-6.01923,1.513975,312.542336,65.909673,1028.88750,0.0
2005-01-01 03:00:00,-0.08866,-5.13102,1.468809,311.980443,68.779475,1028.85440,0.0
2005-01-01 04:00:00,0.32638,-4.65012,1.433129,309.186481,69.211724,1028.79060,0.0
...,...,...,...,...,...,...,...
2024-12-31 19:00:00,3.32736,-3.92136,1.936683,8.167704,58.987088,1025.86625,0.0
2024-12-31 20:00:00,2.54473,-3.64084,1.641096,7.291994,63.677131,1026.36190,0.0
2024-12-31 21:00:00,1.13223,-4.61136,1.292544,5.136621,65.491553,1026.90940,0.0


In [19]:
cams = pd.read_csv(rf"{TMYs_folder}/{site_name}/{site_name}_cams.csv", skiprows=42, sep=';')
cams['datetime'] = pd.to_datetime(cams['# Observation period'].str.split('/').str[0])
cams.set_index('datetime', inplace=True)
cams = cams[['TOA','Clear sky GHI','Clear sky BHI','Clear sky DHI', 'Clear sky BNI', 'GHI','DHI','BNI']]
cams.rename(columns={'BNI': 'DNI'}, inplace=True)

In [20]:
if "CAMS" in selected_files:
    cams.to_csv(
        f"{downloads_dir}/CAMS.csv",
        index=False
    )

In [21]:
update_job('processing_data', 'Processing and merging ERA5 + CAMS datasets...')

[Job #29] processing_data: Processing and merging ERA5 + CAMS datasets...


In [22]:
dataset = pd.concat([cams, era5], axis=1)

In [23]:
dataset.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 175320 entries, 2005-01-01 00:00:00 to 2024-12-31 23:00:00
Data columns (total 15 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   TOA                                175320 non-null  float64
 1   Clear sky GHI                      175320 non-null  float64
 2   Clear sky BHI                      175320 non-null  float64
 3   Clear sky DHI                      175320 non-null  float64
 4   Clear sky BNI                      175320 non-null  float64
 5   GHI                                175263 non-null  float64
 6   DHI                                175263 non-null  float64
 7   DNI                                175263 non-null  float64
 8   Temperature                        175320 non-null  float64
 9   Dew Point                          175320 non-null  float64
 10  Wind Speed                         175320 non-null  float64
 11  Wind Direction  

In [24]:
dataset.to_csv(f'{TMYs_folder}/{site_name}/dataset_{site_name}.csv', index=True)

In [25]:
if "Temperature" in selected_files:
    dataset[['Temperature']].to_csv(
        f"{downloads_dir}/temperature.csv"
    )

if "Wind Speed" in selected_files:
    dataset[['Wind Speed']].to_csv(
        f"{downloads_dir}/wind_speed.csv"
    )

if "Relative Humidity" in selected_files:
    dataset[['Relative Humidity']].to_csv(
        f"{downloads_dir}/humidity.csv"
    )

if "GHI" in selected_files:
    dataset[['GHI']].to_csv(
        f"{downloads_dir}/ghi.csv"
    )

if "DNI" in selected_files:
    dataset[['DNI']].to_csv(
        f"{downloads_dir}/dni.csv"
    )

In [26]:
daily_stats = dataset.resample('D').agg({
    'Temperature': ['min', 'max'],
    'Dew Point': ['min', 'max'],
    'Wind Speed': 'max'
})
daily_stats.columns = ['min_temp', 'max_temp', 'min_dew', 'max_dew', 'max_wind_speed']
daily_stats.reset_index(inplace=True)
dataset = dataset.reset_index().merge(daily_stats, left_on=dataset.index.date, right_on=daily_stats['datetime'].dt.date, how='left')
dataset.drop('key_0', axis=1, inplace=True)

In [27]:
dataset['datetime_x'] = pd.to_datetime(dataset['datetime_x'])
dataset.set_index('datetime_x', inplace=True)

In [28]:
import pvlib
#surface_tilt = 30 
surface_azimuth = 180
location = pvlib.location.Location(latitude=lat, longitude=lon)
solar_position = location.get_solarposition(dataset.index)

poa_irradiance = pvlib.irradiance.get_total_irradiance(
    surface_tilt,
    surface_azimuth,
    solar_position['apparent_zenith'],
    solar_position['azimuth'],
    dataset['DNI'],
    dataset['GHI'],
    dataset['DHI']
)

dataset['GTI'] = poa_irradiance['poa_global']

if "GTI" in selected_files:
    dataset[['GTI']].to_csv(
        f"{downloads_dir}/gti.csv"
    )

path_prepared = f'{TMYs_folder}/{site_name}/prepared_dataset_{site_name}.csv'

dataset.to_csv(path_prepared, index=True)

In [29]:
import zipfile

zip_path = f"{TMYs_folder}/{site_name}/selected_downloads.zip"

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:

    for file in os.listdir(downloads_dir):

        full_path = os.path.join(downloads_dir, file)

        zipf.write(
            full_path,
            arcname=file
        )

print("ZIP created:", zip_path)

ZIP created: TMYs/Talsint-Morocco/selected_downloads.zip


In [30]:
update_job(
    'completed',
    f'Download package created: TMYs/{site_name}/selected_downloads.zip'
)

[Job #29] completed: Download package created: TMYs/Talsint-Morocco/selected_downloads.zip


### Tmy Generation